# WER and CER

Computes Word Error Rate (WER) and Character Error Rate (CER) for Tatar ASR output, comparing transcriptions against gold reference text from a CoNLL file.

This notebook contains two versions:
1. **Basic version** — quick WER/CER computation with lowercasing only, printing the first few comparison examples.
2. **Advanced version** — full text normalization, batch evaluation across multiple result files, and saves mismatches to CSV for error analysis.

In [ ]:
!pip install evaluate
!pip install jiwer

## 1. Basic version

Parses a CoNLL reference file and a raw Söyle ASR output file, then computes WER/CER on lowercased text.

In [ ]:
from evaluate import load


def parse_conll(path):
    references = {}
    current_id = None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# id:"):
                current_id = line.split(":", 1)[1].strip()
            elif line.startswith("# text TAT:"):
                text = line.replace("# text TAT:", "", 1).strip()
                if current_id and text:
                    references[current_id] = text
    return references


def parse_soyle(path):
    predictions = {}
    current_id = None
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# id:"):
                current_id = line.split(":", 1)[1].strip()
            elif line and current_id:
                predictions[current_id] = line
                current_id = None
    return predictions


def compute_metrics(references, predictions):
    wer_metric = load("wer")
    cer_metric = load("cer")

    ids = sorted(set(references.keys()) & set(predictions.keys()))
    refs = [references[i].lower() for i in ids]
    preds = [predictions[i].lower() for i in ids]

    wer = wer_metric.compute(references=refs, predictions=preds)
    cer = cer_metric.compute(references=refs, predictions=preds)
    return wer, cer, ids, refs, preds

In [ ]:
# Set your input file paths
conll_file = "<path-to-tatar-test-conll>"
soyle_file = "<path-to-soyle-results-txt>"

refs = parse_conll(conll_file)
preds = parse_soyle(soyle_file)

wer, cer, ids, refs_list, preds_list = compute_metrics(refs, preds)

print(f"WER: {wer:.4f}")
print(f"CER: {cer:.4f}")
print("\nComparison examples:")
for i, r, p in zip(ids[:10], refs_list[:10], preds_list[:10]):  # first 10 examples
    print(f"{i}:\n  REF: {r}\n  PRD: {p}")

## 2. Advanced version

Adds full text normalization (strips punctuation, lowercases, collapses whitespace), evaluates every `soyle_results_*.txt` file in a folder against the same reference CoNLL file, and saves per-file mismatches to CSV for error analysis, plus an aggregate summary table.

In [ ]:
import os
import re
import pandas as pd
from evaluate import load

wer_metric = load("wer")
cer_metric = load("cer")


# Text normalization
def normalize_text(text: str) -> str:
    # keep letters, digits, and spaces only
    text = re.sub(r"[^\w\s]", "", text, flags=re.UNICODE)
    # lowercase
    text = text.lower()
    # collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_references(conll_path):
    refs = {}
    current_id = None
    with open(conll_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# id:"):
                current_id = line.split(":", 1)[1].strip()
            elif line.startswith("# text TAT:"):
                m = re.match(r"# text TAT:\s*(.*)", line)
                if m and current_id:
                    raw_text = m.group(1).strip()
                    refs[current_id] = {
                        "raw": raw_text,
                        "norm": normalize_text(raw_text)
                    }
    return refs


def load_predictions(txt_path):
    preds = {}
    current_id = None
    with open(txt_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# id:"):
                raw_id = line.split(":", 1)[1].strip().replace(".wav", "")
                if not raw_id.startswith("test_"):
                    raw_id = "test_" + raw_id.split("_")[-1]
                current_id = raw_id
            elif line and not line.startswith("#"):
                if current_id:
                    preds[current_id] = {
                        "raw": line,
                        "norm": normalize_text(line)
                    }
    return preds

In [ ]:
# Compute metrics and save mismatches
def evaluate_file(conll_path, results_path, mismatches_dir="mismatches"):
    references = load_references(conll_path)
    predictions = load_predictions(results_path)

    y_true, y_pred = [], []
    mismatches = []

    for uid, pred in predictions.items():
        if uid in references:
            ref = references[uid]
            y_true.append(ref["norm"])
            y_pred.append(pred["norm"])

            if ref["norm"] != pred["norm"]:
                mismatches.append({
                    "id": uid,
                    "reference_raw": ref["raw"],
                    "prediction_raw": pred["raw"],
                    "reference_norm": ref["norm"],
                    "prediction_norm": pred["norm"]
                })

    wer = wer_metric.compute(references=y_true, predictions=y_pred)
    cer = cer_metric.compute(references=y_true, predictions=y_pred)

    # save mismatches
    os.makedirs(mismatches_dir, exist_ok=True)
    mism_path = os.path.join(
        mismatches_dir, os.path.basename(results_path).replace(".txt", "_mismatches.csv")
    )
    pd.DataFrame(mismatches).to_csv(mism_path, index=False, encoding="utf-8-sig")

    return {
        "file": os.path.basename(results_path),
        "WER": wer,
        "CER": cer,
        "mismatches_file": mism_path
    }

In [ ]:
# Set your input paths
conll_path = "<path-to-tatar-test-conll>"
results_dir = "<path-to-folder-with-soyle-results>"  # folder containing soyle_results_*.txt files

results = []
for fname in os.listdir(results_dir):
    if fname.startswith("soyle_results_") and fname.endswith(".txt"):
        res = evaluate_file(conll_path, os.path.join(results_dir, fname))
        results.append(res)
        print(res)

# aggregate table across files
df = pd.DataFrame(results)
df.to_csv("wer_cer_results.csv", index=False, encoding="utf-8-sig")
print("Summary WER/CER saved to wer_cer_results.csv")